In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv('synthetic_cohorts.csv')

print(f"Loaded {len(df)} rows from systhetic_cohorts.csv")
print(f"\nData shape: {df.shape}")
print(f"Columns : {df.columns.tolist()}")
print(f"\nSample Data:")
print(df.head())

Loaded 1500 rows from systhetic_cohorts.csv

Data shape: (1500, 12)
Columns : ['cohort_month', 'cohort_date', 'channel', 'cac', 'arpu', 'weibull_shape', 'weibull_scale', 'months_since_signup', 'cohort_size', 'active_users', 'retention_rate', 'monthly_revenue']

Sample Data:
  cohort_month cohort_date  channel  cac  arpu  weibull_shape  weibull_scale  \
0      2023-01  2023-01-01  organic   20    15            1.3             25   
1      2023-01  2023-01-01  organic   20    15            1.3             25   
2      2023-01  2023-01-01  organic   20    15            1.3             25   
3      2023-01  2023-01-01  organic   20    15            1.3             25   
4      2023-01  2023-01-01  organic   20    15            1.3             25   

   months_since_signup  cohort_size  active_users  retention_rate  \
0                    0          800           800        1.000000   
1                    1          800           787        0.984886   
2                    2          800  

In [6]:
import numpy as np
import pandas as pd
from scipy.stats import weibull_min

class CohortAnalyzer:
    def __init__(self, cohort_df, discount_rate= 0.10):
        self.df = cohort_df.copy()
        self.discount_rate = discount_rate
        self.monthly_discount_rate = (1+ discount_rate) ** (1/12) - 1
        self._calculate_retention_rates()
        self._calculate_discounted_revenue()

    def _calculate_retention_rates(self):
        self.df['retention_rate'] = (
            self.df['active_users'] / self.df['cohort_size']
        ).fillna(0)

    def _calculate_discounted_revenue(self):
        self.df['discounted_revenue'] = (
            self.df['monthly_revenue'] / ((1+ self.monthly_discount_rate) ** self.df['months_since_signup']))

    def retention_matrix(self, channel = None, metric = 'retention_rate'):
        if channel:
            df= self.df[self.df['channel'] == channel].copy()
        else:
            df = self.df.groupby(['cohort_month', 'months_since_signup']).agg({
                'active_users' : 'sum',
                'cohort_size' : 'sum',
                'retention_rate' : 'mean'
            }).reset_index()

        if metric == 'retention_rate':
            pivot = df.pivot_table(
                index = 'cohort_month',
                columns = 'months_since_signup',
                values = 'active_users'
            )

        else:
            pivot = df.pivot_table(
                index = 'cohort_month',
                columns = 'months_since_signup',
                values = 'active_users'
            )

        return pivot

    def calculate_ltv_by_cohort_channel(self):
        ltv_results= []

        for (cohort, channel), group in self.df.groupby(['cohort_month', 'channel']):
            group = group.sort_values('months_since_signup').reset_index(drop = True)

            cac = group['cac'].iloc[0]
            arpu = group['arpu'].iloc[0]
            ltv = group['discounted_revenue'].sum()


            cumulative = 0
            payback_month = None
            for idx, row in group.iterrows():
                cumulative += row['monthly_revenue']
                if cumulative >= cac and payback_month is None:
                    payback_month = row['months_since_signup']
    
            cum_12m = group[group['months_since_signup'] <= 11]['monthly_revenue'].sum()
            ltv_ratio = round(ltv / cac, 2) if cac > 0 else 0
    
            ltv_results.append({
                'cohort_month' : cohort,
                'channel' : channel,
                'cac' : cac,
                'arpu' : arpu,
                'ltv' : round(ltv, 2),
                'ltv_ratio' : ltv_ratio,
                'payback_month' : payback_month,
                'cumulative_revenue' : round(cum_12m, 2)
            })
    
        return pd.DataFrame(ltv_results)

    def channel_summary(self):

        ltv_cohorts = self.calculate_ltv_by_cohort_channel()

        summary = ltv_cohorts.groupby('channel').agg({
            'cac' : 'first',
            'ltv' : ['mean', 'std'],
            'ltv_ratio' : ['mean', 'std'],
            'payback_month' : ['mean','min','max'],
            'cohort_month' : 'count'
        }).round(2)

        summary.columns = ['_'.join(col).strip('_') for col in summary.columns]
        summary = summary.rename(columns = {'cohort_month_count' : 'cohort_count'})

        return summary.reset_index()


    def payback_analysis(self):
        ltv_cohorts = self.calculate_ltv_by_cohort_channel()

        payback = ltv_cohorts.dropna(subset=['payback_month']).groupby('channel').agg({
            'payback_month' : ['min', 'mean', 'max', 'std']
        }).round(1)

        return payback.reset_index()

    def monthly_retention_curve_by_channel(self, channel):
            channel_data = self.df[self.df['channel'] == channel]
            curve = channel_data.groupby('months_since_signup')['retention_rate'].mean()
            return curve.sort_index()

    def all_retention_curves(self):
        curves = {}
        for channel in self.df['channel'].unique():
            curves[channel] = self.monthly_retention_curve_by_channel(channel)

        return pd.DataFrame(curves)

    def time_to_positive_ltv_distribution(self):
        ltv_cohorts = self.calculate_ltv_by_cohort_channel()
        payback_dist = ltv_cohorts.dropna(subset = ['payback_month'])['payback_month'].value_counts().sort_index()

        return payback_dist

    def profitability_matrix(self):
        ltv_cohorts = self.calculate_ltv_by_cohort_channel()
        matrix = ltv_cohorts.pivot(
            index= 'cohort_month',
            columns = 'channel',
            values = 'ltv_ratio'
        )
        return matrix

    def print_summary_report(self):
        print("COHORT RETENTION & PROFITABILITY ANALYSIS")
        
        print("\n1. CHANNEL SUMMARY (Average LTV by Channel)")
        print("-" * 80)
        channel_summary = self.channel_summary()
        print(channel_summary.to_string(index= False))

        print("\n2. PAYBACK PERIOD ANALYSIS (Months to cover CAC)")
        print("-" * 80)
        payback = self.payback_analysis()
        print(payback.to_string(index = False))

        print("\n3. PAYBACK PERIOD DISTRIBUTION")
        print("-" * 80)
        payback_dist = self.time_to_positive_ltv_distribution()
        for month, count in payback_dist.items():
            print(f" Month {int(month):2d}: {int(count):3d} cohorts")

        print("\n4. COHORT EXTREMES (LTV/CAC Ratio)")
        print("-" * 80)
        ltv_cohorts = self.calculate_ltv_by_cohort_channel()
        best = ltv_cohorts.nlargest(5, 'ltv_ratio')[['cohort_month', 'channel', 'ltv_ratio', 'payback_month']]
        worst = ltv_cohorts.nsmallest(5, 'ltv_ratio')[['cohort_month', 'channel', 'ltv_ratio', 'payback_month']]
        
        print("\nBest Performing Cohorts:")
        print(best.to_string(index= False))
        print("\nWorst Performing Cohorts:")
        print(worst.to_string(index = False))

        print("\n" + "=" * 80 + "\n")                

In [7]:
analyzer = CohortAnalyzer(df, discount_rate = 0.10)

print("CHANNEL SUMMARY\n")
print(analyzer.channel_summary())

CHANNEL SUMMARY

       channel  cac_first   ltv_mean   ltv_std  ltv_ratio_mean  ltv_ratio_std  \
0       direct         35   80869.72  54505.48         2310.56        1557.30   
1      organic         20  124976.45  87942.29         6248.82        4397.12   
2  paid_search         45   79394.37  50739.04         1764.32        1127.53   
3  paid_social         70   33020.29  20418.34          471.72         291.69   
4     referral         25   56531.43  39153.61         2261.26        1566.14   

   payback_month_mean  payback_month_min  payback_month_max  cohort_count  
0                 0.0                  0                  0            22  
1                 0.0                  0                  0            22  
2                 0.0                  0                  0            22  
3                 0.0                  0                  0            22  
4                 0.0                  0                  0            22  


In [8]:
print("PAYBACK ANALYSIS\n")
print(analyzer.payback_analysis())

PAYBACK ANALYSIS

       channel payback_month              
                         min mean max  std
0       direct             0  0.0   0  0.0
1      organic             0  0.0   0  0.0
2  paid_search             0  0.0   0  0.0
3  paid_social             0  0.0   0  0.0
4     referral             0  0.0   0  0.0


In [9]:
print("LTV BY COHORT-CHANNEL- first 20 rows\n")
ltv = analyzer.calculate_ltv_by_cohort_channel()
print(ltv.head(20))

LTV BY COHORT-CHANNEL- first 20 rows

   cohort_month      channel  cac  arpu        ltv  ltv_ratio  payback_month  \
0       2023-01       direct   35    17  231359.75    6610.28              0   
1       2023-01      organic   20    15  370106.34   18505.32              0   
2       2023-01  paid_search   45    16  218684.86    4859.66              0   
3       2023-01  paid_social   70    15   89152.81    1273.61              0   
4       2023-01     referral   25    14  165228.41    6609.14              0   
5       2023-03       direct   35    17  112262.68    3207.51              0   
6       2023-03      organic   20    15  178708.84    8935.44              0   
7       2023-03  paid_search   45    16  106548.33    2367.74              0   
8       2023-03  paid_social   70    15   43489.66     621.28              0   
9       2023-03     referral   25    14   79934.91    3197.40              0   
10      2023-04       direct   35    17  109773.49    3136.39              0   
11

In [10]:
print("RETENTION CURVES BY CHANNEL")
print(analyzer.all_retention_curves())

RETENTION CURVES BY CHANNEL
                     organic  referral  paid_search  paid_social  direct
months_since_signup                                                     
0                    1.00000    1.0000     1.000000     1.000000   1.000
1                    0.98375    0.9800     0.925000     0.870000   0.968
2                    0.96250    0.9525     0.861667     0.786667   0.930
3                    0.93750    0.9225     0.805000     0.716667   0.892
4                    0.91125    0.8925     0.751667     0.660000   0.854
5                    0.88375    0.8600     0.701667     0.606667   0.816
6                    0.85500    0.8275     0.656667     0.560000   0.778
7                    0.82500    0.7975     0.615000     0.520000   0.740
8                    0.79625    0.7650     0.576667     0.483333   0.704
9                    0.76625    0.7325     0.540000     0.450000   0.670
10                   0.73750    0.7025     0.505000     0.420000   0.636
11                   0.

In [11]:
print("PROFITABILITY MATRIX (LTV/CAC Ratio)\n")
print(analyzer.profitability_matrix())

PROFITABILITY MATRIX (LTV/CAC Ratio)

channel        direct   organic  paid_search  paid_social  referral
cohort_month                                                       
2023-01       6610.28  18505.32      4859.66      1273.61   6609.14
2023-03       3207.51   8935.44      2367.74       621.28   3197.40
2023-04       3136.39   8706.92      2322.30       610.04   3120.14
2023-05       6040.08  16674.08      4494.64      1183.04   5988.77
2023-06       2892.80   7938.58      2165.09       571.14   2858.09
2023-07       2800.29   7652.88      2104.82       556.16   2759.75
2023-08       2701.90   7351.03      2040.01       540.12   2655.69
2023-09       2597.13   7032.80      1970.57       523.00   2545.33
2023-10       2485.44   6697.93      1895.81       504.60   2428.58
2023-11       2366.72   6345.52      1815.64       484.69   2304.84
2023-12       2240.44   5974.61      1729.34       463.26   2174.52
2024-01       2106.02   5584.93      1636.81       440.09   2036.99
2024-02   

In [12]:
analyzer.print_summary_report()

COHORT RETENTION & PROFITABILITY ANALYSIS

1. CHANNEL SUMMARY (Average LTV by Channel)
--------------------------------------------------------------------------------
    channel  cac_first  ltv_mean  ltv_std  ltv_ratio_mean  ltv_ratio_std  payback_month_mean  payback_month_min  payback_month_max  cohort_count
     direct         35  80869.72 54505.48         2310.56        1557.30                 0.0                  0                  0            22
    organic         20 124976.45 87942.29         6248.82        4397.12                 0.0                  0                  0            22
paid_search         45  79394.37 50739.04         1764.32        1127.53                 0.0                  0                  0            22
paid_social         70  33020.29 20418.34          471.72         291.69                 0.0                  0                  0            22
   referral         25  56531.43 39153.61         2261.26        1566.14                 0.0               